# Phase 1 — Interactive chess board visualization

**Checkpoint:** `~/models/chesslesson_curriculum_stage/phase1`

走每一步都能看到 board + pieces (SVG 渲染)。每个 turn 展示:
- 起手 board (with pieces + star apples highlighted)
- model 的完整 response (<think> + <action>)
- env step 后新 board (变化看得见)

可以 **改 TASK_ID** 跑不同 lesson:
- `rook-3` (3-move, 3 stars)
- `rook-2` (2-move, 2 stars)
- `castling-1` (1-move 易位)
- `enpassant-1` (1-move 吃过路兵 + scripted opp)
- `checkmate1-1` (1-move 将杀)
- 任何 instructions.jsonl 里的 id


In [ ]:
# Setup
import os, sys, json, re
from pathlib import Path
from IPython.display import display, SVG, HTML, Markdown
import chess, chess.svg

REPO = Path("/home/y50047367/chess_self_play/verl-agent-vam-agent")
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "chess_game" / "chesslesson"))

PHASE = 1
TASK_ID = "rook-3"   # ← 改这里试不同 task
MODEL_PATH = os.path.expanduser(f"~/models/chesslesson_curriculum_stage/phase1")
print(f"Phase {PHASE}  task={TASK_ID}  model={MODEL_PATH}")


In [ ]:
# Load tasks + model
lessons = [json.loads(l) for l in (REPO / "chess_game/chesslesson/instructions.jsonl").open()]
for r in lessons: r["kind"] = "lesson"
TASK = next(l for l in lessons if l["id"] == TASK_ID)
display(Markdown(f"**Task `{TASK_ID}`** — stage={TASK['stage_key']}, budget={TASK['meta']['nbMoves']}, apples={TASK['meta'].get('apples')}"))

from reward import SPECS  # noqa
from stepper import LessonStepper
from chess_game.prompts_shared import build_lesson_initial_obs, build_lesson_step_obs

assert TASK_ID in SPECS, f"{TASK_ID} not in SPECS"


In [ ]:
# Load vLLM model (~30s for 7B)
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
llm = LLM(model=MODEL_PATH, tensor_parallel_size=1, gpu_memory_utilization=0.85,
          max_model_len=8192, trust_remote_code=True, dtype="bfloat16")
sp = SamplingParams(temperature=0.6, top_p=0.95, max_tokens=512, n=1)
print("loaded")


In [ ]:
# Board renderer (highlights apples + last move)
_ACTION_RE = re.compile(r"<action>\s*(.*?)\s*</action>", re.DOTALL | re.IGNORECASE)

def parse_action(text):
    if "<think>" not in text or "</think>" not in text: return ""
    m = _ACTION_RE.search(text)
    return m.group(1).strip().lower().replace(" ", "") if m else ""

def render_board(fen, apples=None, last_move_uci=None, size=380, label=""):
    """Render board SVG with apples highlighted (orange) and last move (green)."""
    b = chess.Board(fen + (" 0 1" if fen.count(" ") < 4 else ""))
    squares = {}
    if apples:
        for sq in apples:
            try: squares[chess.parse_square(sq)] = "#ffae42"  # orange = star/apple
            except Exception: pass
    arrows = []
    if last_move_uci and len(last_move_uci) >= 4:
        try:
            mv = chess.Move.from_uci(last_move_uci)
            arrows = [chess.svg.Arrow(mv.from_square, mv.to_square, color="#2c8d3e")]
        except Exception: pass
    fill = {sq: c for sq, c in squares.items()}
    svg = chess.svg.board(b, size=size, fill=fill, arrows=arrows)
    if label:
        return HTML(f"<div style='display:inline-block;text-align:center'><div style='font-weight:bold;margin-bottom:4px'>{label}</div>{svg}</div>")
    return SVG(svg)

# Test
display(render_board(TASK['fen'], apples=TASK['meta'].get('apples'), label="Starting position (orange = star apples)"))


## Turn 1 — initial state

In [ ]:
# Init stepper, show initial state
stepper = LessonStepper(SPECS[TASK_ID])
print(f"Board fen: {stepper.board_fen()}")
print(f"Items: {sorted(stepper.items)}")
print(f"Opening opponent moves: {list(stepper.opening_opponent)}")

display(render_board(stepper.board_fen(), apples=stepper.items, label="Live position (after opening opp if any)"))


## Run turn-by-turn — each cell call advances 1 turn

In [ ]:
# Helper to run one turn
messages = []
obs = build_lesson_initial_obs(TASK, render_fen=stepper.board_fen(),
                               opening_opponent=list(stepper.opening_opponent) or None)
messages.append({"role": "user", "content": obs})
TURN_COUNTER = [0]

def run_one_turn():
    if stepper.done:
        display(Markdown(f"### ✅ Episode DONE  reward = {1 if stepper.vm.get('completed') else 0}"))
        return
    TURN_COUNTER[0] += 1
    turn = TURN_COUNTER[0]
    
    # Show what user (env) is showing the model this turn
    display(Markdown(f"### Turn {turn}"))
    display(Markdown(f"**Env shows model (user msg):**"))
    print(messages[-1]['content'])
    
    # Render board before move
    display(render_board(stepper.board_fen(), apples=stepper.items, label=f"BEFORE turn {turn} move"))
    
    # Generate
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm.generate([rendered], sp)
    response = out[0].outputs[0].text
    action = parse_action(response)
    messages.append({"role": "assistant", "content": response})
    
    # Show response
    display(Markdown(f"**Model output:**"))
    print(response[:600])
    display(Markdown(f"**Parsed action:** `{action}`"))
    
    # Apply env.step
    pre_items = set(stepper.items)
    pre_hist = len(stepper.chess.history)
    try:
        stepper.step(action)
    except Exception as e:
        display(Markdown(f"⚠️ **Env rejected action**: `{e}`  → episode ENDED with reward 0"))
        return
    post_items = set(stepper.items)
    collected = next(iter(pre_items - post_items)) if (pre_items - post_items) else None
    new_hist = stepper.chess.history[pre_hist:]
    opp_moves = new_hist[1:] if len(new_hist) > 1 else None
    
    # Render board after move
    display(render_board(stepper.board_fen(), apples=stepper.items, last_move_uci=action,
                        label=f"AFTER turn {turn} (green arrow = your move)"))
    print(f"Collected: {collected!r},  remaining: {sorted(stepper.items)}, opp_reply: {opp_moves}")
    
    if not stepper.done and turn < 8:
        next_obs = build_lesson_step_obs(
            render_fen=stepper.board_fen(),
            opponent_moves=opp_moves,
            apples_left=sorted(post_items) if post_items else None,
            collected=collected,
        )
        messages.append({"role": "user", "content": next_obs})
    else:
        display(Markdown(f"### {'✅ SOLVED' if stepper.vm.get('completed') else '❌ FAILED'}  final reward = {1 if stepper.vm.get('completed') else 0}"))

print("Helper ready. Run the next cell to advance turn 1.")


### Turn 1

In [ ]:
run_one_turn()  # turn 1

### Turn 2

In [ ]:
run_one_turn()  # turn 2

### Turn 3

In [ ]:
run_one_turn()  # turn 3

### Turn 4

In [ ]:
run_one_turn()  # turn 4

### Turn 5

In [ ]:
run_one_turn()  # turn 5

### Turn 6

In [ ]:
run_one_turn()  # turn 6

### Turn 7

In [ ]:
run_one_turn()  # turn 7

### Turn 8

In [ ]:
run_one_turn()  # turn 8

---
## Try another task

改 TASK_ID(cell 2)然后从 'Init stepper' cell 重跑即可。